# Generate mini datasett

In [33]:
import os
import sys
import pandas as pd

# Determine the absolute path to the src directory (one level up from notebooks)
module_path = os.path.abspath(os.path.join("..", "src"))
if module_path not in sys.path:
    sys.path.append(module_path)
import plotting
import utils

In [34]:
processed_data_folder = os.path.join(os.path.dirname(os.getcwd()), "data", "processed")
folder_to_minimize = "elec_s_37_ES_PT_no_bat_limit"
folder_to_minimize_path = os.path.join(processed_data_folder, folder_to_minimize)
config_path = os.path.join(folder_to_minimize_path, "config.yaml")
print(folder_to_minimize_path)
output_folder_name = folder_to_minimize + "_mini"
output_folder_path = os.path.join(processed_data_folder, output_folder_name)
new_config_path = os.path.join(output_folder_path, "config.yaml")
print(output_folder_path)

c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\processed\elec_s_37_ES_PT_no_bat_limit
c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\data\processed\elec_s_37_ES_PT_no_bat_limit_mini


In [35]:
if not os.path.exists(output_folder_path):
    os.mkdir(output_folder_path)

In [36]:
num_time_steps = 100

In [37]:
input_data = utils.load_csv_files_from_folder(folder_to_minimize_path)
batteries = input_data["batteries"]
branches = input_data["branches"]
generators = input_data["generators"]
capacity_factors = input_data["capacity_factors"]
generator_costs = input_data["generator_costs"]
hourly_demand = input_data["hourly_demand"]
nodes = input_data["nodes"]
nodes

,x,y,country
bus,,,
ES1 0,-3.427610,40.601332,ES
PT1 0,-8.282125,40.313466,PT


In [38]:
nodes_mini = nodes.copy()
branches_mini = branches.copy()
batteries_mini = batteries.iloc[0:1]  # Just 1 battery

In [39]:
generators_mini = generators[generators["carrier"].isin(["CCGT", "onwind"])]
generators_mini

,bus,carrier,p_nom,marginal_cost,capital_cost,co2_emissions,color,nice_name
generator,,,,,,,,
ES1 0 CCGT,ES1 0,CCGT,26305.332000,40.772380,99027.729293,0.2,#b20101,Combined-Cycle Gas
ES1 0 onwind,ES1 0,onwind,26825.862669,0.015000,96085.888020,0.0,#235ebc,Onshore Wind
PT1 0 CCGT,PT1 0,CCGT,4145.000000,38.876697,99027.729293,0.2,#b20101,Combined-Cycle Gas
PT1 0 onwind,PT1 0,onwind,5213.640675,0.015000,96085.888020,0.0,#235ebc,Onshore Wind


In [40]:
capacity_factors_mini = capacity_factors.iloc[0:num_time_steps]
hourly_demand_mini = hourly_demand.iloc[0:num_time_steps]

In [43]:
nodes_mini.to_csv(os.path.join(output_folder_path, "nodes.csv"))
branches_mini.to_csv(os.path.join(output_folder_path, "branches.csv"))
batteries_mini.to_csv(os.path.join(output_folder_path, "batteries.csv"))
generators_mini.to_csv(os.path.join(output_folder_path, "generators.csv"))
capacity_factors_mini.to_csv(os.path.join(output_folder_path, "capacity_factors.csv"))
hourly_demand_mini.to_csv(os.path.join(output_folder_path, "hourly_demand.csv"))
generator_costs.to_csv(os.path.join(output_folder_path, "generator_costs.csv"))

import shutil

shutil.copy(config_path, new_config_path)

'c:\\Users\\tinus\\OneDrive\\Dokumenter\\0 Master\\code\\master_project\\data\\processed\\elec_s_37_ES_PT_no_bat_limit_mini\\config.yaml'

In [49]:
generators_mini["p_nom"]

generator
ES1 0 CCGT      26305.332000
ES1 0 onwind    26825.862669
PT1 0 CCGT       4145.000000
PT1 0 onwind     5213.640675
Name: p_nom, dtype: float64

In [57]:
total_max_prod = (capacity_factors_mini * generators_mini["p_nom"]).sum().sum()
total_max_prod

3485018.2506259684

In [59]:
total_demand = hourly_demand_mini.sum().sum()
total_demand

3243449.2135653216

In [61]:
def rescale_dataframe(df, target_value):
    """
    Rescale dataframe so that its sum matches the target_value.

    Args:
        df (pd.DataFrame): Input dataframe with numerical values.
        target_value (float): Desired total sum of dataframe values.

    Returns:
        pd.DataFrame: Rescaled dataframe.
    """
    current_total = df.values.sum()
    scaling_factor = target_value / current_total
    return df * scaling_factor

In [64]:
rescale_dataframe(hourly_demand_mini, 1e6)

,ES1 0,PT1 0
snapshot,,
2013-01-01 00:00:00,6933.341806,1464.490327
2013-01-01 01:00:00,6452.890855,1404.985773
2013-01-01 02:00:00,6008.654799,1314.033216
2013-01-01 03:00:00,5729.498311,1233.871640
2013-01-01 04:00:00,5627.191230,1187.624577
...,...,...
2013-01-04 23:00:00,8312.525756,1790.377964
2013-01-05 00:00:00,7603.317379,1617.722262
2013-01-05 01:00:00,6954.165371,1484.222407
